# oxDNA: Propeller Twist Optimization

This notebook optimizes oxDNA1 force-field parameters to match a target propeller twist using the standalone oxDNA simulation engine and DiffTRe for gradient computation.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook optimizes stacking parameters so the simulated propeller twist matches a target value. The propeller twist measures the angle between base normals in hydrogen-bonded pairs — a key structural feature of DNA duplexes. We use the standalone oxDNA binary as the simulation engine, which is the tool most users will already be familiar with.

## Imports

In [2]:
import logging
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md
import mythos.energy.dna1 as dna1_energy
import mythos.input.topology as jdna_top
import mythos.observables as jdna_obs
import mythos.optimization.objective as jdna_objective
import mythos.optimization.optimization as jdna_optimization
import optax
from mythos.input import oxdna_input
from mythos.simulators.oxdna import oxDNASimulator
from mythos.ui.loggers.console import ConsoleLogger
from mythos.utils.units import get_kt_from_string

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO)
logging.getLogger("jax").setLevel(logging.WARNING)

## Configuration

In [ ]:
N_OPT_STEPS = 25
LEARNING_RATE = 5e-4
INPUT_DIR = Path("../../data/full_reparam_oxdna1/60bp_duplex").resolve()
OXDNA_SRC = Path("../../../oxDNA").resolve()
TARGET = jdna_obs.propeller.TARGETS["oxDNA"]

## Load topology and build energy function

In [4]:
top = jdna_top.from_oxdna_file(INPUT_DIR / "sys.top")
sim_config = oxdna_input.read(INPUT_DIR / "input")
kT = get_kt_from_string(sim_config["T"])

with (INPUT_DIR / sim_config["conf_file"]).open("r") as f:
    for line in f:
        if line.startswith("b ="):
            box_size = jnp.array([float(i) for i in line.split("=")[1].strip().split()])
            break

energy_fn = dna1_energy.create_default_energy_fn(
    topology=top,
    displacement_fn=jax_md.space.periodic(box_size)[0],
).with_noopt("ss_stack_weights", "ss_hb_weights", "kt"
).with_params(kt=kT)

transform_fn = energy_fn.energy_fns[0].transform_fn
opt_params = energy_fn.opt_params()

/Users/arik/ws/ssec/mythos/mythos/input/topology.py:251: UserWarning: Type of strand 1 not specified, and did not find T/U for autodetect
  warnings.warn(WARN_CLASSIC_UNSPECIFIED_NT_TYPE.format(strand_idx=i), stacklevel=1)


## Create the simulator

A single `oxDNASimulator` instance pointing at the standalone oxDNA source tree. On each optimization step the binary is recompiled with updated force-field parameters.

In [5]:
simulator = oxDNASimulator(
    name="oxdna-proptwist",
    input_dir=INPUT_DIR,
    energy_fn=energy_fn,
    source_path=OXDNA_SRC,
    input_overrides={
        "steps": 250_000,
        "n_equilibration_steps": 100_000,
        "print_energy_interval": 10_000,
        "print_conf_interval": 10_000,
    },
)

## Define the propeller twist objective

The `PropellerTwist` observable computes the angle between base normals for selected hydrogen-bonded pairs. We pick six internal base pairs from the center of the 60 bp duplex (indices 27–32 on strand 1, paired with 92–87 on strand 2) to avoid end effects.

In [6]:
# Central base pairs: strand-1 index i pairs with strand-2 index (119 - i)
def get_h_bonded_base_pairs(n_nucs_per_strand: int) -> jnp.ndarray:
    s1_nucs = list(range(n_nucs_per_strand))
    s2_nucs = list(range(n_nucs_per_strand, n_nucs_per_strand * 2))
    s2_nucs.reverse()
    return jnp.array(list(zip(s1_nucs, s2_nucs, strict=True)), dtype=jnp.int32)

h_bonded_pairs = get_h_bonded_base_pairs(top.n_nucleotides // 2)

prop_twist_fn = jdna_obs.propeller.PropellerTwist(
    rigid_body_transform_fn=transform_fn,
    h_bonded_base_pairs=h_bonded_pairs,
)

def prop_twist_loss_fn(traj, weights, *_args, **_kwargs):
    obs = prop_twist_fn(traj)
    expected_prop_twist = jnp.dot(weights, obs)
    loss = jnp.sqrt((expected_prop_twist - TARGET) ** 2)
    return loss, (("prop_twist", expected_prop_twist), {})

propeller_twist_objective = jdna_objective.DiffTReObjective(
    name="propeller_twist",
    required_observables=simulator.exposes(),
    logging_observables=["loss", "prop_twist", "neff"],
    grad_or_loss_fn=prop_twist_loss_fn,
    energy_fn=energy_fn,
    min_n_eff_factor=0.95,
    max_valid_opt_steps=100,
)

## Run the optimization

With a single simulator and a single objective, `SimpleOptimizer` is the right
choice. At each step it compiles the updated parameters into a new oxDNA
binary,runs the oxDNA binary, reads the trajectory, and computes DiffTRe
gradients.

In [7]:
opt = jdna_optimization.SimpleOptimizer(
    objective=propeller_twist_objective,
    simulator=simulator,
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    logger=ConsoleLogger(),
)

output = opt.run(params=opt_params, n_steps=N_OPT_STEPS)
print("\nOptimization complete!")
print(f"Final params: {output.opt_params}")

INFO:mythos.simulators.oxdna.oxdna:Updating oxDNA parameters (build path: /var/folders/jf/b0j14xg53sb9dgrsqhn9v1wh0000gn/T/mythos-sim-oxdna-proptwisttd7vgjyc/oxdna-build)
INFO:mythos.simulators.oxdna.oxdna:oxDNA binary rebuilt
INFO:mythos.simulators.oxdna.oxdna:Starting oxDNA simulation
INFO:mythos.simulators.oxdna.oxdna:oxDNA simulation complete


Step: 0, propeller_twist.loss: nan
Step: 0, propeller_twist.neff: nan
Step: 0, propeller_twist.prop_twist: nan


RuntimeError: NaN or Inf detected in gradients at step 0. Is your learning rate too high?